In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import keras
import torch
import numpy as np
import albumentations as A

from pathlib import Path
from typing import Sequence

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from agx_core.transforms import BrightnessAdjustment, LogTransform, GammaCorrection, Deskew


class UnlabeledImageDataset(Dataset):
    def __init__(self, root_dir: Path, cond_shape: Sequence[int], transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.cond_shape = cond_shape
        # Get list of all image file names in the folder
        self.image_files = list(root_dir.glob("*.bmp"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(img_name).convert("L")

        if self.transform:
            image = self.transform(image=np.array(image))
            image = image["image"][..., np.newaxis]

        condition = np.ones(self.cond_shape, dtype=np.float32)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        image = torch.tensor(image, device=device)
        return (image, torch.tensor(condition, device=device)), image


def train_transforms(img_size, mean=[0.5], std=[0.5]):
    return A.Compose(
        [
            Deskew(),
            # A.Pad((25, 25), 255),
            LogTransform(epsilon=2),
            A.InvertImg(1),
            A.Resize(img_size, img_size),
            A.Affine(scale=(0.9, 0.95), rotate=(-90, 90), shear=(5, 5), p=0.5),
            A.RandomRotate90(0.5),
            A.GaussianBlur(blur_range=(1, 3), p=0.3),
            A.Normalize(mean=mean, std=std),
        ]
    )


def valid_transforms(img_size, mean=[0.5], std=[0.5]):
    return A.Compose(
        [
            Deskew(),
            # A.Pad((25, 25), 255),
            LogTransform(epsilon=2),
            A.InvertImg(1),
            A.Resize(img_size, img_size),
            A.Normalize(mean=mean, std=std),
        ]
    )

In [ ]:
img_size = 224
res = img_size // 2**5

img_shape = (None, img_size, img_size, 1)
cond_shape = (None, res, res, 1)

In [ ]:
train_path = Path("../data/LaTuaPastaGlassJars/Clean/train/")
valid_path = Path("../data/LaTuaPastaGlassJars/Clean/val/")
test_path = Path("../data/LaTuaPastaGlassJars/Test/images/train/")

ds_train = UnlabeledImageDataset(
    train_path, transform=train_transforms(img_size), cond_shape=cond_shape[1:]
)
ds_valid = UnlabeledImageDataset(
    valid_path, transform=valid_transforms(img_size), cond_shape=cond_shape[1:]
)
ds_test = UnlabeledImageDataset(
    test_path, transform=valid_transforms(img_size), cond_shape=cond_shape[1:]
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    X, y = ds_test[idx]
    image = X[0].cpu().numpy()
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from keras.optimizers import Adam

from agx_core.models.reversed_autoencoder import (
    MobileNetV3SmallEncoder,
    MobileNetV3SmallDecoder,
)
from agx_torch.models.reversed_autoencoder.model import ReversedAutoencoder

enc = MobileNetV3SmallEncoder(latent_size=512, progressive=True)
dec = MobileNetV3SmallDecoder(target_shape=img_shape[1:], progressive=True)
ra = ReversedAutoencoder(enc, dec, beta_kld=0.1, freeze_backbone=False)
ra.build([img_shape, cond_shape])
ra.place_on_devices("cuda:0", "cuda:1")
ra.compile(Adam(learning_rate=7e-6), Adam(learning_rate=1e-4))

ra.summary()

loader_train = DataLoader(ds_train, batch_size=10, shuffle=True)
loader_valid = DataLoader(ds_valid, batch_size=10)

In [ ]:
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any

rec_dir = Path("reconstructions")
rec_dir.mkdir(exist_ok=True)

plot_every = 50
rec_test_samples = 10
(samples, labels), _ = next(iter(loader_valid))



def plot_on_step_end(step: int, logs: Optional[Dict[str, Any]] = None):
    # plot reconstruction every n epochs/batches
    if (step + 1) % plot_every != 0:
        return

    with torch.no_grad():
        reconstructed = ra([samples, labels]).cpu().numpy()
    samples_resized = ra.resize_progressive_output(samples).cpu().numpy()

    height, width = samples_resized.shape[1:3]
    ar = width / height

    figwidth = 4
    figheight = figwidth / ar

    # 3 columns: original, reconstructed, difference
    fig, axs = plt.subplots(
        rec_test_samples,
        3,
        figsize=(3 * figwidth, rec_test_samples * figheight),
    )

    # Handle single row case
    if rec_test_samples == 1:
        axs = axs.reshape(1, -1)

    for row, (real, rec) in enumerate(zip(samples_resized, reconstructed)):
        rec = np.clip(rec, 0, 1)
        mae = np.abs(rec - real)

        if row == 0:
            axs[row, 0].set_title("Original", fontsize=12, pad=15)
            axs[row, 1].set_title("Reconstruction", fontsize=12, pad=15)
            axs[row, 2].set_title("Anomaly Map", fontsize=12, pad=15)

        # First column: keep axis on for ylabel, hide ticks, rotate 90 degrees
        axs[row, 0].set_ylabel(
            f"{row+1}",
            fontsize=10,
            rotation=90,
            labelpad=15,
            ha="center",
            va="center",
        )

        # All columns: hide ticks
        for col in range(3):
            axs[row, col].set_xticks([])
            axs[row, col].set_yticks([])

        axs[row, 0].imshow(real, cmap="gray", vmax=1)
        axs[row, 1].imshow(rec, cmap="gray", vmax=1)
        axs[row, 2].imshow(mae, cmap="inferno", vmax=1)

    epoch_num = step + 1
    filename = f"epoch_{epoch_num:05d}.jpg"
    title = f"Reconstruction Results - Epoch {epoch_num}"

    # Add overall figure title
    fig.suptitle(title, fontsize=14, y=0.96)

    plt.tight_layout()
    plt.subplots_adjust(top=0.92, left=0.08)

    fig.savefig(rec_dir / filename)

    plt.close(fig)

    print(f"Epoch {step + 1}: reconstruction results saved")

In [ ]:
from keras.callbacks import ModelCheckpoint, LambdaCallback
from agx_core.models.reversed_autoencoder.callbacks import (
    AdversarialEquilibriumCallback,
    ProgressiveGrowingCallback,
)

callbacks = [
    AdversarialEquilibriumCallback(0.3, -0.5, min_pause_steps=100),
    ProgressiveGrowingCallback(2000, 2000),
    ModelCheckpoint(
        filepath="ra_mbnetv3.best.keras",
        monitor="val_loss_rec",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    LambdaCallback(on_epoch_end=plot_on_step_end),
]

In [ ]:
history = ra.fit(
    loader_train,
    validation_data=loader_valid,
    epochs=10000,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
ra.save("ra_mbnetv3.model.keras")

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(history.history)

hist_file = Path("history.csv")
if hist_file.exists():
    hist = pd.read_csv(hist_file)
    df.index += len(hist)
    hist = pd.concat([hist, df])
    # hist.to_csv(hist_file, index=False)
else:
    df.to_csv(hist_file, index=False)
    hist = df
hist.tail()

In [ ]:
import keras
import matplotlib.pyplot as plt

ra = keras.models.load_model("ra_mbnetv3.best.keras", safe_mode=False)
while not ra.decoder.is_fully_grown:
    ra.encoder.grow()
    ra.encoder.alpha = 1
    ra.decoder.grow()
    ra.decoder.alpha = 1


fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    (I, C), y = ds_train[idx]
    rec = ra([I[np.newaxis, ...], C[np.newaxis, ...]])
    ax.imshow(rec.cpu().detach().numpy()[0], cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

fig, ax = plt.subplots(figsize=(21, 14))

# Adjust val_loss_embed, forgot to update validation scaling
hist[["kld_fake", "kld_real"]].plot(ax=ax)

In [ ]:
with torch.device("cpu"):
    ra.eval()
    I = torch.rand((1, *img_shape[1:]))
    C = torch.rand((1, *cond_shape[1:]))
    Z = torch.rand((1, *cond_shape[1:3], 512))

    import keras

    i_ = keras.Input(img_shape[1:])
    c_ = keras.Input(cond_shape[1:])

    (mean, logvar), _ = ra.encoder([i_, c_])
    z = ra.reparameterize([mean, logvar])
    rec = ra.decoder([z, c_])

    model = keras.Model(
        inputs=[i_, c_], outputs=[rec, mean, logvar], name="reversed_autoencoder"
    )
    model.training = False
    model.eval()

    torch.onnx.export(
        model,
        ([I, C],),
        "model.onnx",
        input_names=["image", "condition"],
        output_names=["reconstruction", "mean", "logvar"],
    )

In [ ]:
import matplotlib.pyplot as plt

from agx_core.helpers import _channel_axis
from keras import ops


def kl_divergence(mean, logvar):
    return 0.5 * ops.sum(
        ops.square(mean) + ops.exp(logvar) - logvar - 1.0,
        axis=_channel_axis(),
    )


fig, axes = plt.subplots(5, 3, figsize=(12, 20))

for idx in range(5):
    (I, C), y = ds_test[idx]
    rec = ra_best([I[np.newaxis, ...], C[np.newaxis, ...]]).cpu().detach().numpy()[0][0]
    (mean, logvar), _ = ra_best.encoder([I[np.newaxis, ...], C[np.newaxis, ...]])
    kld = kl_divergence(mean, logvar).cpu().detach().numpy()[0]
    I = I[0].cpu().detach().numpy()
    D = np.square(I - rec)
    axes.flat[3 * idx].imshow(rec, cmap="gray")
    axes.flat[3 * idx + 1].imshow(I, cmap="gray")
    axes.flat[3 * idx + 2].imshow(D, cmap="gray")
    axes.flat[3 * idx].axis("off")
    axes.flat[3 * idx + 1].axis("off")
    axes.flat[3 * idx + 2].axis("off")

plt.tight_layout()
plt.show()